In [ ]:
# ============================================================
# SUPERCONDUCTOR CRITICAL TEMPERATURE PREDICTION
# DECISION TREE REGRESSION
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("train.csv")

print("="*60)
print("DATASET INFORMATION")
print("="*60)

print("Shape :", df.shape)

# ============================================================
# TARGET VARIABLE
# ============================================================

target = "critical_temp"

# ============================================================
# INPUT AND OUTPUT
# ============================================================

X = df.drop(columns=[target])
y = df[target]

# ============================================================
# HANDLE MISSING VALUES
# ============================================================

X = X.fillna(X.median())
y = y.fillna(y.median())

# ============================================================
# FEATURE SELECTION USING CORRELATION
# ============================================================

correlation = X.corrwith(y)

correlation_table = pd.DataFrame({
    "Feature": X.columns,
    "Correlation": correlation,
    "Absolute_Correlation": abs(correlation)
})

correlation_table = correlation_table.sort_values(
    by="Absolute_Correlation",
    ascending=False
)

print("\nTop Correlated Features")
print(correlation_table.head(20))

# ============================================================
# SELECT IMPORTANT FEATURES
# ============================================================

threshold = 0.10

important_features = correlation_table[
    correlation_table["Absolute_Correlation"] >= threshold
]["Feature"].tolist()

print("\nNumber of Important Features:",
      len(important_features))

X = X[important_features]

# ============================================================
# REMOVE MULTICOLLINEAR FEATURES
# ============================================================

corr_matrix = X.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

drop_features = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.90)
]

print("\nFeatures Removed due to Multicollinearity:")
print(drop_features)

X = X.drop(columns=drop_features)

print("\nFinal Feature Count:", X.shape[1])

# ============================================================
# FINAL CORRELATION MATRIX
# ============================================================

final_df = pd.concat([X, y], axis=1)

plt.figure(figsize=(14,10))

sns.heatmap(
    final_df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title(
    "Correlation Matrix of Selected Features"
)

plt.show()

# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ============================================================
# FEATURE SCALING
# (Not required for Decision Trees, but kept for consistency
#  with the original pipeline / in case you compare models)
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# ============================================================
# DECISION TREE REGRESSION MODEL
# ============================================================

model = DecisionTreeRegressor(
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

model.fit(
    X_train_scaled,
    y_train
)

# ============================================================
# PREDICTIONS
# ============================================================

y_pred = model.predict(X_test_scaled)

# ============================================================
# EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    y_pred
)

mape = mean_absolute_percentage_error(
    y_test,
    y_pred
)

n = len(y_test)
p = X_test.shape[1]

adj_r2 = 1 - (
    ((1-r2)*(n-1))
    /
    (n-p-1)
)

print("\n")
print("="*60)
print("MODEL PERFORMANCE")
print("="*60)

print(f"MAE           : {mae:.4f}")
print(f"MSE           : {mse:.4f}")
print(f"RMSE          : {rmse:.4f}")
print(f"R2 Score      : {r2:.4f}")
print(f"Adjusted R2   : {adj_r2:.4f}")
print(f"MAPE          : {mape:.4f}")

# ============================================================
# ACTUAL VS PREDICTED
# ============================================================

plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.7
)

plt.xlabel("Actual Critical Temperature")
plt.ylabel("Predicted Critical Temperature")

plt.title(
    "Actual vs Predicted"
)

plt.show()

# ============================================================
# RESIDUAL PLOT
# ============================================================

residuals = y_test - y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals,
    alpha=0.7
)

plt.axhline(
    y=0,
    color='red'
)

plt.xlabel("Predicted Values")
plt.ylabel("Residuals")

plt.title(
    "Residual Plot"
)

plt.show()

# ============================================================
# FEATURE IMPORTANCE
# (Decision Trees use feature_importances_, not coefficients)
# ============================================================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n")
print("="*60)
print("FEATURE IMPORTANCE")
print("="*60)

print(importance)

# ============================================================
# FEATURE IMPORTANCE GRAPH
# ============================================================

plt.figure(figsize=(10,8))

plt.barh(
    importance["Feature"],
    importance["Importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")

plt.title(
    "Feature Importance (Decision Tree)"
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()

# ============================================================
# TREE VISUALIZATION (optional, top few levels)
# ============================================================

plt.figure(figsize=(20,10))

plot_tree(
    model,
    max_depth=3,
    feature_names=X.columns,
    filled=True,
    fontsize=8
)

plt.title("Decision Tree (first 3 levels)")

plt.show()